In [6]:
import math

def ring_edges(n: int):
    return [(i, i + 1) for i in range(n - 1)] + [(n - 1, 0)]

def brickwork_ring_edges(n: int):
    """
    Return ring edges in a brickwork-style order:
    even bonds, odd bonds, then closing bond if needed.
    For n=32, this gives a nearest-neighbor layered ordering.
    """
    even_edges = [(i, i + 1) for i in range(0, n - 1, 2)]
    odd_edges = [(i, i + 1) for i in range(1, n - 1, 2)]
    closing = [(n - 1, 0)]
    return even_edges + odd_edges + closing

def write_qaoa32_maxcut_qasm(
    filename: str = "QAOA32.qasm",
    n: int = 32,
    p: int = 2,
    gamma = None,
    beta = None,
    use_brickwork: bool = True,
):
    """
    Generate a standard QAOA-MaxCut circuit on a 32-qubit nearest-neighbor ring.

    Structure:
      - H on all qubits
      - p repetitions of:
          cost layer: CX-RZ-CX for each graph edge
          mixer layer: RX on all qubits

    This is the best-supported candidate structure for the paper's QAOA32 benchmark:
      - nearest-neighbor application
      - deeper than GHZ
      - still local, unlike random long-range stress circuits
    """

    if gamma is None:
        gamma = [0.6] * p
    if beta is None:
        beta = [0.4] * p

    if len(gamma) != p or len(beta) != p:
        raise ValueError("gamma and beta must both have length p")

    edges = brickwork_ring_edges(n) if use_brickwork else ring_edges(n)

    with open(filename, "w", encoding="utf-8") as f:
        f.write("OPENQASM 2.0;\n")
        f.write('include "qelib1.inc";\n')
        f.write(f"qreg q[{n}];\n")

        # Initial layer: |+>^n
        for q in range(n):
            f.write(f"h q[{q}];\n")

        # QAOA layers
        for layer in range(p):
            g = gamma[layer]
            b = beta[layer]

            # Cost layer for MaxCut: exp(-i * gamma * Z_i Z_j)
            # Decomposition: CX - RZ(2*gamma) - CX
            for u, v in edges:
                f.write(f"cx q[{u}],q[{v}];\n")
                f.write(f"rz({2.0 * g:.12f}) q[{v}];\n")
                f.write(f"cx q[{u}],q[{v}];\n")

            # Mixer layer: exp(-i * beta * X)
            for q in range(n):
                f.write(f"rx({2.0 * b:.12f}) q[{q}];\n")

    print(f"Wrote {filename}")
    print(f"Qubits: {n}")
    print(f"Layers p: {p}")
    print(f"Edges per cost layer: {len(edges)}")
    print(f"Total CX count: {2 * len(edges) * p}")


if __name__ == "__main__":
    write_qaoa32_maxcut_qasm(
        filename="QAOA32.qasm",
        n=32,
        p=2,
        gamma=[0.60, 0.45],
        beta=[0.40, 0.30],
        use_brickwork=True,
    )


Wrote QAOA32.qasm
Qubits: 32
Layers p: 2
Edges per cost layer: 32
Total CX count: 128


In [ ]:
import math

def generate_sqrt30_qasm():
    n = 30
    # 1. 计算 Grover 迭代次数: floor(pi/4 * sqrt(2^n))
    N = 2**n
    nstep = math.floor((math.pi / 4) * math.sqrt(N))
    
    # 定义寄存器名
    # a: 搜索寄存器 (30 bits)
    # b: 平方结果寄存器 (30 bits)
    # t: 目标标记位 (1 bit)
    # anc: 辅助位 (29 bits, 对应 Scaffold 中的 x[n-1])
    
    print('OPENQASM 2.0;')
    print('include "qelib1.inc";')
    print('')
    print(f'qreg a[{n}];')
    print(f'qreg b[{n}];')
    print(f'qreg t[1];')
    print(f'qreg anc[{n-1}];')
    print(f'creg ma[{n}];')
    print(f'creg mt[1];')
    print('')

    # --- 定义内部函数以保持逻辑一致 ---

    def sqr():
        """实现 b(x) = a(x)^2 mod (x^30 + x + 1)"""
        # Part 1: i <= 14
        for i in range(0, (n - 1) // 2 + 1):
            k = 2 * i
            print(f'cx a[{i}], b[{k}];')
        # Part 2: i >= 15
        for i in range((n + 1) // 2, n):
            k = (2 * i) - n
            print(f'cx a[{i}], b[{k}];')
            print(f'cx a[{i}], b[{k+1}];')

    def eqxmark(tF):
        """测试 b(x) == x. 逻辑: b == 0100...0 (只有b[1]=1)"""
        # 1. 翻转除 b[1] 以外的所有位，使得检测目标变为全 1
        for j in range(n):
            if j != 1:
                print(f'x b[{j}];')
        
        # 2. 计算 AND 链 (Toffoli 链)
        print(f'ccx b[1], b[0], anc[0];')
        for j in range(1, n - 1):
            print(f'ccx anc[{j-1}], b[{j+1}], anc[{j}];')
        
        # 3. 标记
        if tF != 0:
            print(f'cx anc[{n-2}], t[0];') # 返回到 t
        else:
            print(f'z anc[{n-2}];')        # 相位翻转
            
        # 4. 逆运算撤销 AND 链
        for j in range(n - 2, 0, -1):
            print(f'ccx anc[{j-1}], b[{j+1}], anc[{j}];')
        print(f'ccx b[1], b[0], anc[0];')

        # 5. 还原 b
        for j in range(n):
            if j != 1:
                print(f'x b[{j}];')

    def diffuse():
        """扩散算子"""
        # 1. Hadamard 和 X
        for j in range(n):
            print(f'h a[{j}];')
            print(f'x a[{j}];')
        
        # 2. 准备辅助位 (PrepZ 在 QASM 中默认已是 0，故不重复初始化)
        # 计算 AND 链检测是否全为 0 (已被 X 翻转为 1)
        print(f'ccx a[1], a[0], anc[0];')
        for j in range(1, n - 1):
            print(f'ccx anc[{j-1}], a[{j+1}], anc[{j}];')
            
        # 3. 相位翻转
        print(f'z anc[{n-2}];')
        
        # 4. 撤销 AND 链
        for j in range(n - 2, 0, -1):
            print(f'ccx anc[{j-1}], a[{j+1}], anc[{j}];')
        print(f'ccx a[1], a[0], anc[0];')

        # 5. 还原 X 和 Hadamard
        for j in range(n):
            print(f'x a[{j}];')
            print(f'h a[{j}];')

    # --- 主程序逻辑 ---

    # 1. 初始化 a 进入均匀叠加态
    for i in range(n):
        print(f'h a[{i}];')

    # 2. Grover 迭代循环
    print(f'// 开始 Grover 迭代，共 {nstep} 次')
    for istep in range(1, nstep + 1):
        print(f'// Iteration {istep}')
        sqr()
        eqxmark(0)
        sqr() # Sqr 在 GF(2) 下是自身的逆
        diffuse()

    # 3. 最终测量准备
    sqr()
    eqxmark(1)

    # 4. 测量
    print('mt[0] = measure t[0];')
    for i in range(n):
        print(f'ma[{i}] = measure a[{i}];')

if __name__ == "__main__":
    generate_sqrt30_qasm()


In [1]:
import os
import numpy as np
from qiskit.qasm2 import dump
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import efficient_su2

np.random.seed(42)
path = "./programs"


def Adder(n_qubits: int, use_toffoli: bool = False, save_qasm: bool = False) -> QuantumCircuit:
    if n_qubits % 2 != 0:
        raise ValueError(f"n_qubits must be even, but got {n_qubits}")
    n_bits = n_qubits // 2 - 1
    circuit = QuantumCircuit(n_qubits)

    def _toffoli(x: int, y: int, z: int):
        circuit.h(z)
        circuit.cx(y, z)
        circuit.tdg(z)
        circuit.cx(x, z)
        circuit.t(z)
        circuit.cx(y, z)
        circuit.t(y)
        circuit.tdg(z)
        circuit.cx(x, z)
        circuit.cx(x, y)
        circuit.t(z)
        circuit.h(z)
        circuit.t(x)
        circuit.tdg(y)
        circuit.cx(x, y)

    def _MAJ(x: int, y: int, z: int, use_toffoli: bool):
        circuit.cx(z, y)
        circuit.cx(z, x)
        if use_toffoli:
            circuit.ccx(x, y, z)
        else:
            _toffoli(x, y, z)

    def _UMA(x: int, y: int, z: int, use_toffoli: bool):
        circuit.x(y)
        circuit.cx(x, y)
        if use_toffoli:
            circuit.ccx(x, y, z)
        else:
            _toffoli(x, y, z)
        circuit.x(y)
        circuit.cx(z, x)
        circuit.cx(z, y)

    ind = [2 * i + 2 for i in range(n_bits)]
    for i in ind:
        _MAJ(i - 2, i - 1, i, use_toffoli)
    circuit.cx(ind[-1], n_qubits - 1)
    for i in reversed(ind):
        _UMA(i - 2, i - 1, i, use_toffoli)

    if save_qasm:
        filename = os.path.join(path, f"Adder{n_qubits}.qasm")
        dump(circuit, filename)
    return circuit


def BV(n_qubits: int, save_qasm: bool = False) -> QuantumCircuit:
    circuit = QuantumCircuit(n_qubits)
    circuit.x(n_qubits - 1)
    circuit.h(range(n_qubits))
    for i in range(n_qubits - 1):
        circuit.cx(i, n_qubits - 1)
    circuit.h(range(n_qubits))

    if save_qasm:
        filename = os.path.join(path, f"BV{n_qubits}.qasm")
        dump(circuit, filename)
    return circuit


def GHZ(n_qubits: int, save_qasm: bool = False) -> QuantumCircuit:
    circuit = QuantumCircuit(n_qubits)
    circuit.h(0)
    for i in range(n_qubits - 1):
        circuit.cx(i, i + 1)

    if save_qasm:
        filename = os.path.join(path, f"GHZ{n_qubits}.qasm")
        dump(circuit, filename)
    return circuit


def QAOA_HEA(n_qubits: int, entanglement: str, reps: int, save_qasm: bool = False) -> QuantumCircuit:
    """entanglement= "full", "linear", "reverse_linear", "pairwise", "circular", or "sca"
    https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.circuit.library.TwoLocal"""
    circuit = efficient_su2(n_qubits, entanglement=entanglement, reps=reps)
    params = np.random.uniform(-np.pi, np.pi, size=circuit.num_parameters)
    circuit = circuit.assign_parameters(params)
    if save_qasm:
        filename = os.path.join(path, f"QAOA_HEA{n_qubits}_{entanglement}_{reps}.qasm")
        dump(circuit, filename)
    return circuit


def QFT(n_qubits: int, do_swap: bool = False, save_qasm: bool = False) -> QuantumCircuit:
    circuit = QuantumCircuit(n_qubits)
    for i in range(n_qubits):
        circuit.h(i)
        for j in range(i + 1, n_qubits):
            angle = np.pi / (2 ** (j - i))
            circuit.cp(angle, j, i)
    circuit = transpile(circuit, basis_gates=["h", "rz", "cx"])

    if do_swap:
        for i in range(n_qubits // 2):
            circuit.swap(i, n_qubits - i - 1)

    if save_qasm:
        filename = os.path.join(path, f"QFT{n_qubits}.qasm")
        dump(circuit, filename)
    return circuit


n_qubits = 32
circuit = QAOA_HEA(n_qubits, entanglement="circular", reps=20, save_qasm=True)
print(circuit.count_ops())
# circuit.draw("mpl", fold=0)

OrderedDict({'ry': 672, 'rz': 672, 'cx': 640})
